In [ ]:
# CPU allocation / notebook configuration
# Edit CPU_LIST to a comma-separated list or range string such as "0-7,12".
import os

CPU_LIST = os.environ.get("NOTEBOOK_CPU_LIST", "")

def parse_cpu_list(text):
    cpus = []
    for part in str(text).split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            left, right = part.split('-', 1)
            cpus.extend(range(int(left), int(right) + 1))
        else:
            cpus.append(int(part))
    return sorted(set(cpus))

_selected_cpus = parse_cpu_list(CPU_LIST)
if _selected_cpus and hasattr(os, 'sched_setaffinity'):
    os.sched_setaffinity(0, set(_selected_cpus))

for _name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_name, str(max(1, len(_selected_cpus) or 1)))
print("CPU affinity:", sorted(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else 'unavailable')


# Finite Lyapunov Mode Spatial Analysis

This notebook analyzes saved `finite_lyapunov_modes` from the GPU alpha-sweep data. The mode vectors are in the reduced Choi slab basis; `lyapunov_mode_basis_indices` maps them back to full top-layer indices `mu + 2*x + 2*Nx*y`.

In [ ]:
from __future__ import annotations

import json, math, re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    mpl.rcParams.update({
        "font.family": "CMU Sans Serif",
        "mathtext.fontset": "cm",
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.8,
    })
except Exception:
    pass

ROOT = Path.cwd()
if ROOT.name != "class_A_fermionic_adaptive_circuit":
    candidate = ROOT.parent
    if (candidate / "colab_no_feedback_alpha_sweep_transfer").exists():
        ROOT = candidate
DATA_ROOT = ROOT / "colab_no_feedback_alpha_sweep_transfer" / "gpu_data"
OUTPUT_ROOT = ROOT / "colab_no_feedback_alpha_sweep_transfer" / "analysis_outputs" / "gpu_data_summary" / "lyapunov_modes"
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
for path in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

RUN_RE = re.compile(r"N(?P<Nx>\d+)x(?P<Ny>\d+)_dwtrunc(?P<dw>\d+)_corr_(?P<corr>none|perfect)_alpha_(?P<atag>.+)$")
FIG_WIDTH = 3.375
COLORS = {20: "#1f77b4", 30: "#2ca02c", 40: "#d62728"}
CORR_LABEL = {"none": "No feedback", "perfect": "Perfect correction"}
LS = {"none": "-", "perfect": "--"}

def parse_alpha_tag(tag): return float(str(tag).replace('p', '.'))
def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(path)
    return path

print("DATA_ROOT", DATA_ROOT)
print("OUTPUT_ROOT", OUTPUT_ROOT)


In [ ]:
def full_index_to_components(indices, Nx):
    indices = np.asarray(indices, dtype=np.int64)
    mu = indices % 2
    site = indices // 2
    x = site % int(Nx)
    y = site // int(Nx)
    return x.astype(np.int64), y.astype(np.int64), mu.astype(np.int64)


def circular_mean_and_width(y_vals, weights, period):
    weights = np.asarray(weights, dtype=np.float64)
    if weights.sum() <= 0:
        return np.nan, np.nan
    theta = 2.0 * np.pi * np.asarray(y_vals, dtype=np.float64) / float(period)
    z = np.sum(weights * np.exp(1j * theta)) / np.sum(weights)
    mean_theta = np.angle(z)
    if mean_theta < 0:
        mean_theta += 2.0 * np.pi
    mean_y = mean_theta * float(period) / (2.0 * np.pi)
    R = np.abs(z)
    circ_std = np.sqrt(max(0.0, -2.0 * np.log(max(R, 1e-15)))) * float(period) / (2.0 * np.pi)
    return float(mean_y), float(circ_std)


def mode_metrics(vector, basis_indices, Nx, Ny):
    x, y, mu = full_index_to_components(basis_indices, Nx)
    amp = np.abs(np.asarray(vector, dtype=np.complex128)) ** 2
    total = float(amp.sum())
    if not np.isfinite(total) or total <= 0:
        return None
    w = amp / total
    site_density = np.zeros((int(Nx), int(Ny)), dtype=np.float64)
    np.add.at(site_density, (x, y), w)
    x_density = site_density.sum(axis=1)
    y_density = site_density.sum(axis=0)
    nonzero_x = np.where(x_density > 1e-12)[0]
    x_min = int(nonzero_x.min()) if nonzero_x.size else int(x.min())
    x_max = int(nonzero_x.max()) if nonzero_x.size else int(x.max())
    edge_dist = np.minimum(np.abs(np.arange(int(Nx)) - x_min), np.abs(np.arange(int(Nx)) - x_max))
    x_mean = float(np.sum(np.arange(int(Nx)) * x_density))
    x_width = float(np.sqrt(np.sum(((np.arange(int(Nx)) - x_mean) ** 2) * x_density)))
    y_mean, y_width = circular_mean_and_width(np.arange(int(Ny)), y_density, int(Ny))
    ipr_site = float(np.sum(site_density ** 2))
    pr_site = float(1.0 / ipr_site) if ipr_site > 0 else np.nan
    edge_weight_0 = float(x_density[x_min] + x_density[x_max]) if x_min != x_max else float(x_density[x_min])
    edge_mask_1 = edge_dist <= 1
    edge_weight_1 = float(np.sum(x_density[edge_mask_1]))
    edge_weight_2 = float(np.sum(x_density[edge_dist <= 2]))
    mean_edge_dist = float(np.sum(edge_dist * x_density))
    left_weight = float(x_density[x_min])
    right_weight = float(x_density[x_max])
    edge_asym = float((right_weight - left_weight) / max(right_weight + left_weight, 1e-15))
    center_x = 0.5 * (x_min + x_max)
    center_weight = float(np.sum(x_density[np.abs(np.arange(int(Nx)) - center_x) <= 1.0]))
    return {
        "norm": total,
        "x_min": x_min,
        "x_max": x_max,
        "slab_width_sites": x_max - x_min + 1,
        "x_mean": x_mean,
        "x_width": x_width,
        "y_mean_circular": y_mean,
        "y_width_circular": y_width,
        "site_ipr": ipr_site,
        "site_participation": pr_site,
        "site_participation_fraction": pr_site / max(1, np.count_nonzero(site_density > 1e-12)),
        "edge_weight_0": edge_weight_0,
        "edge_weight_1": edge_weight_1,
        "edge_weight_2": edge_weight_2,
        "mean_edge_distance": mean_edge_dist,
        "left_edge_weight": left_weight,
        "right_edge_weight": right_weight,
        "edge_asymmetry": edge_asym,
        "center_weight": center_weight,
        "x_density": x_density,
        "site_density": site_density,
    }


In [ ]:
mode_rows = []
run_profile_acc = {}
gap_profile_acc = {}
rep_modes = []
bad = []

sample_paths = sorted(DATA_ROOT.rglob('runs/*/samples/sample_*.npz'))
for sample_path in sample_paths:
    run_dir = sample_path.parents[1]
    match = RUN_RE.match(run_dir.name)
    if not match:
        continue
    gd = match.groupdict()
    Nx, Ny = int(gd['Nx']), int(gd['Ny'])
    corr = gd['corr']
    alpha = parse_alpha_tag(gd['atag'])
    try:
        with np.load(sample_path, allow_pickle=False) as z:
            if 'finite_lyapunov_modes' not in z or 'lyapunov_mode_basis_indices' not in z:
                continue
            vectors = np.asarray(z['finite_lyapunov_modes'], dtype=np.complex128)
            exps = np.asarray(z['finite_lyapunov_exponents'], dtype=np.float64)
            a_vals = np.asarray(z['finite_lyapunov_a_eigenvalues'], dtype=np.float64) if 'finite_lyapunov_a_eigenvalues' in z else np.full(exps.shape, np.nan)
            basis = np.asarray(z['lyapunov_mode_basis_indices'], dtype=np.int64)
            sample_index = int(z['sample_index'])
            if vectors.ndim != 2 or vectors.shape[1] != exps.size:
                bad.append({'path': str(sample_path), 'issue': f'mode shape {vectors.shape} exponent size {exps.size}'})
                continue
            key = (Nx, Ny, corr, alpha)
            if key not in run_profile_acc:
                run_profile_acc[key] = {'x_density_sum': np.zeros(Nx), 'count': 0}
                gap_profile_acc[key] = {'x_density_sum': np.zeros(Nx), 'site_density_sum': np.zeros((Nx, Ny)), 'count': 0}
            gap_pos = int(np.argmin(np.abs(exps))) if exps.size else None
            for mode_pos in range(exps.size):
                metrics = mode_metrics(vectors[:, mode_pos], basis, Nx, Ny)
                if metrics is None:
                    continue
                run_profile_acc[key]['x_density_sum'] += metrics['x_density']
                run_profile_acc[key]['count'] += 1
                is_gap_mode = bool(mode_pos == gap_pos)
                if is_gap_mode:
                    gap_profile_acc[key]['x_density_sum'] += metrics['x_density']
                    gap_profile_acc[key]['site_density_sum'] += metrics['site_density']
                    gap_profile_acc[key]['count'] += 1
                    if alpha in (1.0, 1.5, 1.8, 1.9, 2.0, 2.2) and sample_index == 0:
                        rep_modes.append({
                            'Nx': Nx, 'Ny': Ny, 'correction': corr, 'alpha': alpha,
                            'sample_index': sample_index, 'mode_position': mode_pos,
                            'exponent': float(exps[mode_pos]),
                            'site_density': metrics['site_density'],
                            'x_density': metrics['x_density'],
                        })
                row = {k: v for k, v in metrics.items() if k not in ('x_density', 'site_density')}
                row.update({
                    'Nx': Nx,
                    'Ny': Ny,
                    'correction': corr,
                    'alpha': alpha,
                    'sample_index': sample_index,
                    'mode_position': mode_pos,
                    'finite_lyapunov_exponent': float(exps[mode_pos]),
                    'abs_finite_lyapunov_exponent': float(abs(exps[mode_pos])),
                    'finite_lyapunov_a_eigenvalue': float(a_vals[mode_pos]) if mode_pos < a_vals.size else np.nan,
                    'is_gap_mode': is_gap_mode,
                    'sample_path': str(sample_path.relative_to(ROOT)),
                })
                mode_rows.append(row)
    except Exception as exc:
        bad.append({'path': str(sample_path), 'issue': repr(exc)})

mode_metrics_df = pd.DataFrame(mode_rows)
bad_df = pd.DataFrame(bad)
print('sample files', len(sample_paths), 'mode rows', len(mode_metrics_df), 'bad', len(bad_df))
display(mode_metrics_df.head())


In [ ]:
# Aggregate per run and save tables.
if mode_metrics_df.empty:
    raise RuntimeError('No Lyapunov mode rows were parsed.')

agg_df = (
    mode_metrics_df.groupby(['Nx','Ny','correction','alpha'], as_index=False)
    .agg(
        mode_count=('mode_position','count'),
        sample_count=('sample_index','nunique'),
        mean_abs_exponent=('abs_finite_lyapunov_exponent','mean'),
        sem_abs_exponent=('abs_finite_lyapunov_exponent', lambda s: float(s.std(ddof=1)/np.sqrt(len(s))) if len(s)>1 else 0.0),
        mean_site_participation=('site_participation','mean'),
        sem_site_participation=('site_participation', lambda s: float(s.std(ddof=1)/np.sqrt(len(s))) if len(s)>1 else 0.0),
        mean_participation_fraction=('site_participation_fraction','mean'),
        mean_x_width=('x_width','mean'),
        mean_y_width=('y_width_circular','mean'),
        mean_edge_weight_0=('edge_weight_0','mean'),
        mean_edge_weight_1=('edge_weight_1','mean'),
        mean_edge_weight_2=('edge_weight_2','mean'),
        mean_edge_distance=('mean_edge_distance','mean'),
        mean_center_weight=('center_weight','mean'),
        mean_edge_asymmetry=('edge_asymmetry','mean'),
    )
)

gap_mode_df = mode_metrics_df[mode_metrics_df['is_gap_mode']].copy()
gap_agg_df = (
    gap_mode_df.groupby(['Nx','Ny','correction','alpha'], as_index=False)
    .agg(
        sample_count=('sample_index','nunique'),
        gap_mode_abs_exponent_mean=('abs_finite_lyapunov_exponent','mean'),
        gap_mode_abs_exponent_sem=('abs_finite_lyapunov_exponent', lambda s: float(s.std(ddof=1)/np.sqrt(len(s))) if len(s)>1 else 0.0),
        gap_mode_participation_mean=('site_participation','mean'),
        gap_mode_participation_sem=('site_participation', lambda s: float(s.std(ddof=1)/np.sqrt(len(s))) if len(s)>1 else 0.0),
        gap_mode_edge_weight_1_mean=('edge_weight_1','mean'),
        gap_mode_edge_distance_mean=('mean_edge_distance','mean'),
        gap_mode_x_width_mean=('x_width','mean'),
        gap_mode_center_weight_mean=('center_weight','mean'),
    )
)

profile_rows = []
for key, acc in sorted(run_profile_acc.items()):
    Nx, Ny, corr, alpha = key
    if acc['count']:
        mean_x = acc['x_density_sum'] / acc['count']
        for x, value in enumerate(mean_x):
            profile_rows.append({'Nx':Nx,'Ny':Ny,'correction':corr,'alpha':alpha,'profile_kind':'all_modes_mean_x','x':x,'density':float(value),'count':acc['count']})
for key, acc in sorted(gap_profile_acc.items()):
    Nx, Ny, corr, alpha = key
    if acc['count']:
        mean_x = acc['x_density_sum'] / acc['count']
        for x, value in enumerate(mean_x):
            profile_rows.append({'Nx':Nx,'Ny':Ny,'correction':corr,'alpha':alpha,'profile_kind':'gap_modes_mean_x','x':x,'density':float(value),'count':acc['count']})
profile_df = pd.DataFrame(profile_rows)

mode_metrics_df.to_csv(TABLE_DIR / 'lyapunov_mode_metrics.csv', index=False)
agg_df.to_csv(TABLE_DIR / 'lyapunov_mode_aggregate.csv', index=False)
gap_mode_df.to_csv(TABLE_DIR / 'gap_lyapunov_mode_metrics.csv', index=False)
gap_agg_df.to_csv(TABLE_DIR / 'gap_lyapunov_mode_aggregate.csv', index=False)
profile_df.to_csv(TABLE_DIR / 'lyapunov_mode_x_profiles.csv', index=False)
bad_df.to_csv(TABLE_DIR / 'lyapunov_mode_parse_issues.csv', index=False)

np.savez_compressed(
    OUTPUT_ROOT / 'lyapunov_mode_profiles.npz',
    profile_records=profile_df.to_records(index=False),
    aggregate_records=agg_df.to_records(index=False),
    gap_aggregate_records=gap_agg_df.to_records(index=False),
)

display(agg_df.head())
display(gap_agg_df.head())


In [ ]:
# Figure: participation ratio of all finite Lyapunov modes.
fig, axes = plt.subplots(1, 2, figsize=(2*FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ['none','perfect']):
    for Ny, sub in agg_df[agg_df.correction == corr].groupby('Ny'):
        sub = sub.sort_values('alpha')
        ax.errorbar(sub.alpha, sub.mean_site_participation, yerr=sub.sem_site_participation, color=COLORS.get(int(Ny)), marker='o', lw=1.0, ms=3, label=f'Ny={int(Ny)}')
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r'$\alpha_1$')
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('site participation ratio')
axes[1].legend(frameon=False)
savefig('lyapunov_participation_vs_alpha.png')
plt.show()


In [ ]:
# Figure: gap-mode participation ratio.
fig, axes = plt.subplots(1, 2, figsize=(2*FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ['none','perfect']):
    for Ny, sub in gap_agg_df[gap_agg_df.correction == corr].groupby('Ny'):
        sub = sub.sort_values('alpha')
        ax.errorbar(sub.alpha, sub.gap_mode_participation_mean, yerr=sub.gap_mode_participation_sem, color=COLORS.get(int(Ny)), marker='o', lw=1.0, ms=3, label=f'Ny={int(Ny)}')
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r'$\alpha_1$')
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('gap-mode site participation')
axes[1].legend(frameon=False)
savefig('gap_mode_participation_vs_alpha.png')
plt.show()


In [ ]:
# Figure: edge weight within one column of the slab edges.
fig, axes = plt.subplots(1, 2, figsize=(2*FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ['none','perfect']):
    for Ny, sub in gap_agg_df[gap_agg_df.correction == corr].groupby('Ny'):
        sub = sub.sort_values('alpha')
        ax.plot(sub.alpha, sub.gap_mode_edge_weight_1_mean, color=COLORS.get(int(Ny)), marker='o', lw=1.0, ms=3, label=f'Ny={int(Ny)}')
    ax.set_ylim(0, 1.02)
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r'$\alpha_1$')
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('gap-mode edge weight, dist <= 1')
axes[1].legend(frameon=False)
savefig('gap_mode_edge_weight_vs_alpha.png')
plt.show()


In [ ]:
# Figure: mean distance of gap modes from slab edge.
fig, axes = plt.subplots(1, 2, figsize=(2*FIG_WIDTH, 2.6), sharey=True)
for ax, corr in zip(axes, ['none','perfect']):
    for Ny, sub in gap_agg_df[gap_agg_df.correction == corr].groupby('Ny'):
        sub = sub.sort_values('alpha')
        ax.plot(sub.alpha, sub.gap_mode_edge_distance_mean, color=COLORS.get(int(Ny)), marker='o', lw=1.0, ms=3, label=f'Ny={int(Ny)}')
    ax.set_title(CORR_LABEL[corr])
    ax.set_xlabel(r'$\alpha_1$')
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('mean distance from slab edge')
axes[1].legend(frameon=False)
savefig('gap_mode_edge_distance_vs_alpha.png')
plt.show()


In [ ]:
# Figure: average x-density profiles of gap modes for representative alpha values.
representative_alphas = [1.0, 1.5, 1.8, 1.9, 2.0, 2.2]
fig, axes = plt.subplots(2, 3, figsize=(3*FIG_WIDTH, 4.8), sharey=True)
for ax, alpha in zip(axes.ravel(), representative_alphas):
    sub = profile_df[(profile_df.profile_kind == 'gap_modes_mean_x') & np.isclose(profile_df.alpha, alpha)]
    for (Ny, corr), group in sub.groupby(['Ny','correction']):
        group = group.sort_values('x')
        ax.plot(group.x, group.density, color=COLORS.get(int(Ny)), ls=LS[corr], lw=1.0, label=f'Ny={int(Ny)}, {corr}')
    ax.set_title(rf'$\alpha_1={alpha:g}$')
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.2)
axes[0,0].set_ylabel('gap-mode density')
axes[1,0].set_ylabel('gap-mode density')
axes[0,2].legend(frameon=False, fontsize=6)
savefig('gap_mode_x_density_profiles.png')
plt.show()


In [ ]:
# Figure: all-mode mean x-density profiles for representative alpha values.
representative_alphas = [1.0, 1.5, 1.8, 1.9, 2.0, 2.2]
fig, axes = plt.subplots(2, 3, figsize=(3*FIG_WIDTH, 4.8), sharey=True)
for ax, alpha in zip(axes.ravel(), representative_alphas):
    sub = profile_df[(profile_df.profile_kind == 'all_modes_mean_x') & np.isclose(profile_df.alpha, alpha)]
    for (Ny, corr), group in sub.groupby(['Ny','correction']):
        group = group.sort_values('x')
        ax.plot(group.x, group.density, color=COLORS.get(int(Ny)), ls=LS[corr], lw=1.0, label=f'Ny={int(Ny)}, {corr}')
    ax.set_title(rf'$\alpha_1={alpha:g}$')
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.2)
axes[0,0].set_ylabel('all-mode mean density')
axes[1,0].set_ylabel('all-mode mean density')
axes[0,2].legend(frameon=False, fontsize=6)
savefig('all_mode_x_density_profiles.png')
plt.show()


In [ ]:
# Figure: representative gap-mode spatial heatmaps.
choices = [
    (40, 'none', 1.9),
    (30, 'none', 1.5),
    (40, 'perfect', 1.5),
    (20, 'perfect', 1.5),
]
fig, axes = plt.subplots(1, len(choices), figsize=(len(choices)*FIG_WIDTH, 2.8), constrained_layout=True)
for ax, (Ny, corr, alpha) in zip(axes, choices):
    candidates = [r for r in rep_modes if r['Ny'] == Ny and r['correction'] == corr and np.isclose(r['alpha'], alpha)]
    if candidates:
        dens = candidates[0]['site_density'].T
        im = ax.imshow(dens, origin='lower', aspect='auto', cmap='magma')
        ax.set_title(f"Ny={Ny}, {corr}\n" + rf"$\alpha_1={alpha:g}$")
        ax.set_xlabel('x')
        ax.set_ylabel('y')
    else:
        ax.text(0.5, 0.5, 'missing', ha='center', va='center')
        ax.set_axis_off()
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, label='site density')
out = FIG_DIR / 'representative_gap_mode_heatmaps.png'
fig.savefig(out, bbox_inches='tight', dpi=300)
print(out)
plt.show()


In [ ]:
# Write executive summary JSON.
small_pr = gap_agg_df.sort_values(['Ny','correction','gap_mode_participation_mean']).groupby(['Ny','correction'], as_index=False).first()
edge_max = gap_agg_df.sort_values(['Ny','correction','gap_mode_edge_weight_1_mean'], ascending=[True,True,False]).groupby(['Ny','correction'], as_index=False).first()
summary = {
    'mode_row_count': int(len(mode_metrics_df)),
    'gap_mode_row_count': int(len(gap_mode_df)),
    'run_profile_count': int(len(agg_df)),
    'parse_issue_count': int(len(bad_df)),
    'mean_metrics_by_correction': agg_df.groupby('correction').agg({
        'mean_site_participation':'mean',
        'mean_edge_weight_1':'mean',
        'mean_edge_distance':'mean',
        'mean_x_width':'mean',
        'mean_y_width':'mean',
        'mode_count':'sum',
    }).reset_index().to_dict(orient='records'),
    'gap_mode_smallest_participation_by_size_correction': small_pr.to_dict(orient='records'),
    'gap_mode_largest_edge_weight_by_size_correction': edge_max.to_dict(orient='records'),
    'figure_paths': sorted(str(p.relative_to(ROOT)) for p in FIG_DIR.glob('*.png')),
    'table_paths': sorted(str(p.relative_to(ROOT)) for p in TABLE_DIR.glob('*.csv')),
}
(OUTPUT_ROOT / 'lyapunov_mode_executive_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True)[:6000])
